In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from torchinfo import summary

# Processing pipeline in Transformers

---

## 🗺️ The Processing Pipeline

When a sentence passes into a Transformer, it goes through this specific chain:

```text
Raw Text:          "The cat sits"
                        │
Step 1: Tokenizer  (Converts text to Vocabulary Integer IDs)
                        ▼
Token IDs:         [ 101,  2043,  4112 ]  -> Shape: (Batch, Seq_Len)
                        │
Step 2: nn.Embedding  (Converts IDs to dense float vectors)
                        ▼
Token Embeddings:  [ [0.21, -0.4], [0.9, 0.1], ... ] -> Shape: (Batch, Seq_Len, d_model)
                        │
Step 3: Positional Encoding (Adds sequence order information)
                        ▼
Final Input:       Ready for Encoder/Decoder attention blocks!
```

---

We first introduce several components in transformers.

##  1. Tokenization

To process text data, we need to convert strings into numbers.

### 1.1. The Industry Standard: Hugging Face `tokenizers` / `transformers`
99% of modern Deep Learning setups use Hugging Face tokenizers outside of PyTorch. These tokenizers are written in Rust, run in parallel across multiple CPU cores, and output native PyTorch tensors seamlessly via a single line of code.

### 1.2. The Legacy Way: `torchtext` (Deprecating/Archived)
PyTorch used to maintain an official text-domain repository called `torchtext`. It provided regex-based word splitters (like `basic_english`). However, PyTorch has officially archived this library, and it is no longer supported for modern LLM development.

### 1.3. Pure Python Custom Lookup (For Toy/Character Models)
If you are building an educational model (like an architectural scratch-build of a Character-GPT), you can write a simple vocabulary dictionary using native Python primitives and convert the integers into tensors manually.

---

### 💻 Code Demonstration: The 2 Main Approaches

#### Option A: The Real-World Standard (Hugging Face + PyTorch)
This is how modern text-processing pipelines are structured. 

The transformers library is required:

```
    pip install transformers datasets sacremoses
```

In [ ]:
from transformers import AutoTokenizer

# 1. External specialized library handles string-to-integer mapping
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Text converts directly into native PyTorch Tensors ('return_tensors="pt"')
text = "PyTorch processes tensors, not string characters."
model_inputs = tokenizer(text, return_tensors="pt")

print("--- APPROACH A: INDUSTRY STANDARD ---")

print ("Input text:", text)
print("PyTorch Input IDs Tensor:")
print(model_inputs["input_ids"])

# 3. Stream immediately into standard PyTorch Network layers
embedding_layer = nn.Embedding(tokenizer.vocab_size, 16)
dense_vectors = embedding_layer(model_inputs["input_ids"])
print("\nShape after PyTorch nn.Embedding:", dense_vectors.shape)
# Shape: (batch_size=1, tokens=10, d_model=16)

We can try different texts.

In [ ]:
print (tokenizer.vocab_size)

text = "This is cool!"
out = tokenizer(text, return_tensors="pt")
print (out["input_ids"])

text = "123 This is cool"
out = tokenizer(text, return_tensors="pt")
print (out["input_ids"])

text = "This is"
out = tokenizer(text, return_tensors="pt")
print (out["input_ids"])

text = "This"
out = tokenizer(text, return_tensors="pt")
print (out["input_ids"])

#### Option B: The Native Scratch Character Tokenizer
If you are restricted from using third-party packages, this shows how you manage a minimalist character-level vocabulary using pure Python combined with `torch.tensor`:


In [ ]:
# 1. Set up a simple localized corpus
alphabet = "abcdefghijklmnopqrstuvwxyz "
char_list = sorted(list(set(alphabet)))

# 2. Map every character to a specific number and vice versa
char_to_id = { ch:i for i, ch in enumerate(char_list) }
id_to_char = { i:ch for i, ch in enumerate(char_list) }

# 3. Simple lambda to turn a string into a list of numbers
encode_string = lambda text_string: [char_to_id[char] for char in text_string]

# 4. Convert the basic Python list into a long integer PyTorch Tensor
sample_word = "pytorch"
integer_ids = encode_string(sample_word)
pytorch_tensor = torch.tensor(integer_ids, dtype=torch.long)

print("\n--- APPROACH B: CUSTOM NATIVE ENCODER ---")
print(f"Original Text: '{sample_word}'")
print("Custom Generated PyTorch Tensor:")
print(pytorch_tensor)

## 2. nn.Embedding

https://docs.pytorch.org/docs/2.12/generated/torch.nn.Embedding.html

A Transformer cannot accept raw token integer IDs directly. `nn.Embedding` is a mandatory layer that acts as a trainable dictionary, converting integer token IDs into dense numerical vectors.

Below is an example.


In [ ]:
# an Embedding module containing 10 tensors of size 3
embedding = nn.Embedding(num_embeddings=10, embedding_dim=3)

# a batch of 2 samples of 4 indices each
input = torch.LongTensor([[1, 2, 4, 5], [4, 3, 2, 9]])

# each of the integers from 0 to 9 is mapped to a 3d vector. 
output = embedding(input)

print (output.shape, "\n", output)

It contains trainalbe parameters.

In [ ]:
summary(embedding)

## 3. Positional Encoding in Transformers

**Positional Encoding** is a fundamental component of Transformer models. Because the underlying Self-Attention mechanism processes data completely in parallel, it possesses **no inherent sense of order or sequence information**. Without this encoding, the sentence *"The dog chases the cat"* would have the exact same mathematical representation as *"The cat chases the dog"*. To inject word order, a position vector of the same dimension is added directly to each token vector (word embedding).

---

### Sinusoidal Encoding

In the original paper *"Attention Is All You Need"*, the authors utilized a fixed, deterministic mathematical formula based on **sine and cosine functions** with varying frequencies:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)
$$
$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right)
$$

*   **$pos$**: The absolute position of the word in the sentence (0, 1, 2, ...).
*   **$i$**: The index of the dimension within the embedding vector.
*   **$d_{model}$**: The total dimension of the model (e.g., 512).

#### Advantages:
1.  **Uniqueness:** Every position receives a unique, continuous signal.
2.  **Bounded Values:** Values remain stable within the range \([-1, 1]\).
3.  **Relative Relationships:** Due to trigonometric identities, the model can learn to map linear translations ($pos + k$ can be expressed as a linear function of $pos$).

Below is the concret implementation.

In [ ]:
# =====================================================================
# 1. POSITIONAL ENCODING
# =====================================================================
class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, max_len: int = 5000):
        """
        Classic sinusoidal positional encoding from 'Attention Is All You Need'.
        
        Args:
            d_model: The embedding dimension of the transformer (e.g., 512).
            max_len: The maximum sequence length supported.
        """
        super().__init__()
        
        # Create a matrix of shape (max_len, d_model) filled with zeros
        pe = torch.zeros(max_len, d_model)
        
        # Position vector: shape (max_len, 1) -> [0, 1, 2, ...]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Scaling factor for frequencies (calculated for even indices only)
        # Mathematical identity: 10000^(2i / d_model) -> exp(2i * -log(10000) / d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices (0, 2, 4, ...)
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices (1, 3, 5, ...)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Expand dimensions for batching: shape (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # 'register_buffer' saves this matrix as part of the module state,
        # but prevents it from being tracked as a trainable gradient parameter.
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        """
        Args:
            x: Input token embeddings tensor of shape (batch_size, seq_len, d_model)
        Returns:
            Tensor with positional encodings added.
        """
        # Slice the pre-computed encodings to match the current input sequence length
        x = x + self.pe[:, :x.size(1)]
        return x

In essence, it shifts the data using a constant tensor.

We can visualize the positional vector.  

In [ ]:
pos = PositionalEncoding(512, max_len=100)

print (pos.pe.shape)

fig = plt.figure(figsize=(6,3))
plt.imshow(pos.pe[0], aspect='auto')
plt.xlabel('depth i')
plt.ylabel('position')
plt.colorbar()
plt.title("PE")
plt.tight_layout()
#fig.savefig("transformer_pe.png")

Combine the two embeddings together

In [ ]:
# an Embedding module containing 10 tensors of size 100
embedding_dim = 100

embedding = nn.Embedding(num_embeddings=10, embedding_dim=embedding_dim)

# a batch of 2 samples of 4 indices each
input = torch.LongTensor([[1, 2, 4, 5], [4, 3, 2, 9]])

# Each of the integers from 0 to 9 is mapped to a 100d vector. 
output = embedding(input)

print ("input shape:", input.shape)

print ("\nafter embedding:", output.shape)

pos = PositionalEncoding(embedding_dim, 100)

pos_output = pos(output)

print ("\nafter positional embedding:", pos_output.shape)

## 4. Scaled Dot-Product Attention

**Scaled Dot-Product Attention** is the mathematical core of the Transformer architecture. It allows a model to dynamically score the relationships between different words in a sequence, enabling tokens to aggregate contextual information from the most relevant parts of the text.

---

### 🧮 The Mathematical Formula

The entire operation is expressed as a highly optimized matrix computation:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

#### Core Components:
*   **Query ($Q$):** Represents the *current* token looking for context (the "question").
*   **Key ($K$):** Represents *all* tokens in the sequence being searched (the "index" or "tags").
*   **Value ($V$):** Represents the *actual content* or semantic meaning of the tokens (the "information").
*   **$d_k$:** The dimensionality of the Keys/Queries (the size of each individual attention head).

---

### 🪜 Step-by-Step Execution Mechanics

#### Step 1: Compute Similarity (Dot Product)
The model multiplies the Query matrix by the transposed Key matrix ($QK^T$). This calculates a raw alignment score between every single pair of tokens. 
*   **High score:** The words are highly relevant to each other.
*   **Low score:** The words have little to no relation.

#### Step 2: Scale the Scores ($\frac{1}{\sqrt{d_k}}$)
The raw scores are divided by the square root of the key dimension ($\sqrt{d_k}$). 
*   **Why?** As the dimension $d_k$ grows large, the dot products grow very large in magnitude. This pushes the softmax function into regions with extremely small gradients (vanishing gradients), which completely stalls model training. Scaling stabilizes the mathematical variance.

#### Step 3: Normalize (Softmax)
The scaled scores pass through a row-wise Softmax function. This converts the raw scores into formal probabilities between `0` and `1` that sum up to exactly `1`. These percentages represent the **Attention Weights**.

#### Step 4: Aggregate Information ($\cdot V$)
Finally, the attention weights are multiplied by the Value matrix ($V$). Tokens with high attention percentages have their Value vectors heavily emphasized, while irrelevant tokens are filtered out.

---

### Dimensions Cheat Sheet

Let's define the shape variables:
* **B**: Batch size (number of sequences processed at the same time)
* **T**: Sequence length (number of tokens/words in the input)
* **d_k**: Dimensionality of Queries and Keys
* **d_v**: Dimensionality of Values


| Step / Tensor | Mathematical Component | Input Shapes | Output Shape |
| :--- | :--- | :--- | :--- |
| **Queries** | $Q$ | — | `(B, T, d_k)` |
| **Keys (Transposed)** | $K^T$ | — | `(B, d_k, T)` |
| **Values** | $V$ | — | `(B, T, d_v)` |
| **Dot Product Scores** | $Q K^T$ | `(B, T, d_k) × (B, d_k, T)` | `(B, T, T)` |
| **Attention Weights** | $\text{softmax}(\dots)$ | `(B, T, T)` | `(B, T, T)` |
| **Final Context Block** | $\text{Weights} \cdot V$ | `(B, T, T) × (B, T, d_v)` | `(B, T, d_v)` |


Below is the Implementation.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor = None):
        """
        Args:
            Q: Query tensor of shape (batch_size, n_heads, seq_len, head_dim)
            K: Key tensor of shape (batch_size, n_heads, seq_len, head_dim)
            V: Value tensor of shape (batch_size, n_heads, seq_len, head_dim)
            mask: Optional tensor containing 0s (to mask out) and 1s (to keep)
        """
        # Extract head_dim (d_k) from the last dimension of the Key tensor
        d_k = K.size(-1)
        
        # Step 1 & 2: Calculate raw scores and scale them
        # Shape: (batch_size, n_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Step 3 (Optional): Apply Masking (e.g., Causal or Padding Mask)
        if mask is not None:
            # Replaces masked positions with -inf so they become 0 after softmax
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        # Step 4: Apply Softmax to get normalized probabilities
        attention_weights = F.softmax(scores, dim=-1)
        
        # Step 5: Multiply weights by Values to get final context representation
        # Shape: (batch_size, n_heads, seq_len, head_dim)
        context = torch.matmul(attention_weights, V)
        
        return context, attention_weights

# --- Verification ---
# Hyperparameters: Batch=1, Heads=2, Sequence Length=3, Head Dimension=4
b, h, s, d = 1, 2, 3, 4

attention_layer = ScaledDotProductAttention()

# Generate dummy Q, K, V matrices
dummy_Q = torch.randn(b, h, s, d)
dummy_K = torch.randn(b, h, s, d)
dummy_V = torch.randn(b, h, s, d)

output, weights = attention_layer(dummy_Q, dummy_K, dummy_V)

print("Scores/Weights Matrix Shape:", weights.shape) # Expects: (1, 2, 3, 3)
print("Final Context Output Shape: ", output.shape)  # Expects: (1, 2, 3, 4)
print("\nAttention Weights (Head 0):\n", weights)

## 5. Multi-Head Attention (MHA) 

**Multi-Head Attention** is the core powerhouse of the Transformer architecture. Instead of computing attention across the entire embedding dimension all at once, Multi-Head Attention splits the embedding vector into multiple smaller sub-spaces ("heads"). This allows the model to simultaneously focus on information from different parts of a sentence at different levels of abstraction.

---

### 💡 The Intuition: Why use multiple heads?

If a model only uses a single attention head, it can only focus on **one relationship at a time**. 

Consider the sentence: *"The animal didn't cross the street because it was too tired."*
*   **Head 1** might learn that **"it"** refers to **"The animal"** (Noun-Pronoun relationship).
*   **Head 2** might learn that **"tired"** explains why it didn't **"cross"** (Verb-Adjective relationship).
*   **Head 3** might track structural grammar rules (Subject-Object relationship).

By splitting the representation into \(H\) heads, the model can capture all of these distinct semantic relationships in parallel.

---

### 🧮 The Step-by-Step Mathematical Mechanics

The process splits the original embedding dimension ($d_{model}$) into $H$ heads, where each head has a smaller dimensionality: 
$$d_k = \frac{d_{model}}{H}$$

#### Step 1: Linear Projections
The input sequence matrix $X$ is multiplied by trainable weight matrices to create individual Query ($Q$), Key ($K$), and Value ($V$) projections for every single head ($i$):
$$
Q_i = X \cdot W_i^Q, \quad K_i = X \cdot W_i^K, \quad V_i = X \cdot W_i^V
$$

#### Step 2: Scaled Dot-Product Attention (Per Head)
Each head independently computes attention scores. The scores are scaled by $\sqrt{d_k}$ to prevent gradients from exploding during softmax:
$$
\text{head}_i = \text{softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}}\right) V_i
$$

#### Step 3: Concatenation
The outputs of all individual attention heads are glued back together horizontally. This restores the matrix back to the original model dimension ($d_{model}$):
$$\text{Concat} = [\text{head}_1; \text{head}_2; \dots; \text{head}_H]$$

#### Step 4: Final Linear Projection
The concatenated matrix is passed through one final linear layer ($W^O$) to allow the information from all the different heads to mix together completely:
$$\text{MultiHead}(Q, K, V) = \text{Concat} \cdot W^O$$

In PyTorch, the Multi-head attention mechanism is implemented in **nn.MulitheadAttention** class.

https://docs.pytorch.org/docs/2.12/generated/torch.nn.MultiheadAttention.html


Below is an example.

In [ ]:
BATCH_SIZE = 2
SEQ_LEN = 5     # Number of tokens in the sequence
D_MODEL = 32    # Embedding dimension (must be divisible by NHEAD)
NHEAD = 4       # Number of attention heads

# Initialize the Multihead Attention layer
# batch_first=True expects tensors shaped as (Batch, Seq, Features)
mha = nn.MultiheadAttention(embed_dim=D_MODEL, num_heads=NHEAD, batch_first=True)

# Generate random dummy data mimicking embedded text
# Shape: [Batch_Size, Seq_Len, d_model]
query = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)
key   = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)
value = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)

# ==========================================
# 2. CREATING ATTENTION MASKS
# ==========================================

# A. Causal Mask (For Decoders / Future Token Blocking)
# Shape: [Seq_Len, Seq_Len]
# We use boolean masks: True means MASK/IGNORE, False means KEEP
causal_mask = torch.triu(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool), diagonal=1)

# B. Padding Mask (For ignoring PAD tokens in a batch)
# Shape: [Batch_Size, Seq_Len]
# True means the token is padding and should be ignored
key_padding_mask = torch.tensor([
    [False, False, False, False, False], # Sentence 1 has no padding
    [False, False, False,  True,  True]  # Sentence 2 has 2 padding tokens at the end
], dtype=torch.bool)

# ==========================================
# 3. FORWARD PASS
# ==========================================

# PyTorch's MHA returns two things:
# 1. The calculated context output tensor
# 2. The averaged attention weights matrix (optional, set average_attn_weights=False to get raw weights)
attn_output, attn_weights = mha(
    query=query, 
    key=key, 
    value=value, 
    attn_mask=causal_mask, 
    key_padding_mask=key_padding_mask
)

# ==========================================
# 4. VERIFY SHAPES
# ==========================================
print("--- Tensor Shapes ---")
print(f"Input Query Shape:  {query.shape}")         # [2, 5, 32]
print(f"Output Attn Shape:  {attn_output.shape}")   # [2, 5, 32] -> Matches Query Shape
print(f"Attn Weights Shape: {attn_weights.shape}")  # [2, 5, 5]  -> [Batch, Query_Len, Key_Len]


### Direct implementation

We can implement the attention mechanism ourselves. 

This code demonstrates the internal math of a single Encoder layer, including

1. Self-Attention
2. Residual Connections
3. Layer Normalization
4. Feed-Forward Network

---

### Multi-Head Attention Dimension Tracking

Here is the precise dimension tracking for **Multi-Head Attention (MHA)** with batching.

Let's define the shape variables:
* **B**: Batch size
* **T**: Sequence length (number of tokens)
* **d_model**: Total hidden dimensionality of the model (e.g., 512)
* **H**: Number of attention heads (e.g., 8)
* **d_k**: Dimensionality per head for Queries/Keys ($d_k = d_{\text{model}} / H$, e.g., 64)
* **d_v**: Dimensionality per head for Values ($d_v = d_{\text{model}} / H$, e.g., 64)

---

### Step-by-Step Dimension Pipeline

#### 1. Input Projections
The sequence representation $X$ of shape `(B, T, d_model)` is projected into Q, K, and V using linear layers:
* **Query (Q)**: `(B, T, d_model)`
* **Key (K)**: `(B, T, d_model)`
* **Value (V)**: `(B, T, d_model)`

#### 2. Reshaping for Multi-Head Splitting
To compute attention for all heads in parallel, the final dimension is split into $H \times d_k$, and the tensor is transposed:
* **Reshape**: Split `(B, T, d_model)` into `(B, T, H, d_k)`
* **Transpose**: Permute the head dimension to the front -> **`(B, H, T, d_k)`**
* *Applies to all inputs:*
  * **$Q$ Shape**: `(B, H, T, d_k)`
  * **$K$ Shape**: `(B, H, T, d_k)`
  * **$V$ Shape**: `(B, H, T, d_v)`

#### 3. Transposing Keys for Scaled Dot-Product ($K^T$)
The last two dimensions of the Key matrix are flipped within each head:
* **$K^T$ Shape**: `(B, H, d_k, T)`

#### 4. Multi-Head Score Calculation ($Q \cdot K^T$)
Batch matrix multiplication happens across the last two dimensions for every head independently:
* Operation: `(B, H, T, d_k) × (B, H, d_k, T)`
* **Result Shape**: `(B, H, T, T)` *(Attention scores per head)*

#### 5. Softmax Activation
Applying `softmax` normalizes the scores across the last dimension, preserving the structure:
* **Attention Weights Shape**: `(B, H, T, T)`

#### 6. Multi-Head Context Block ($\text{Weights} \times V$)
The attention weights are multiplied by the split Value matrix:
* Operation: `(B, H, T, T) × (B, H, T, d_v)`
* **Result Shape**: `(B, H, T, d_v)`

#### 7. Concatenation and Final Projection
To return to the original model structure, the heads are merged back together and passed through a final linear layer ($W^O$):
* **Transpose**: Permute back to `(B, T, H, d_v)`
* **Flatten (Concat)**: Combine the last two dimensions -> `(B, T, H * d_v)` which equals **`(B, T, d_model)`**
* **Output Projection Layer**: `(B, T, d_model) × (d_model, d_model)`
* **Final Output Shape**: `(B, T, d_model)`

---

### Multi-Head Attention Cheat Sheet


| Step / Tensor | Mathematical Operation / Shape Logic | Dimension |
| :--- | :--- | :--- |
| **Input Token Embeddings** | Input Matrix $X$ | `(B, T, d_model)` |
| **Split & Transposed Inputs** | Permute Head Dimension to Index 1 | `(B, H, T, d_k)` |
| **Transposed Keys ($K^T$)** | Permute last two axes of $K$ | `(B, H, d_k, T)` |
| **Raw Attention Scores** | $Q \times K^T$ | `(B, H, T, T)` |
| **Attention Weights** | $\text{softmax}(\text{Scores} / \sqrt{d_k})$ | `(B, H, T, T)` |
| **Head Context Output** | $\text{Weights} \times V$ | `(B, H, T, d_v)` |
| **Concatenated Heads** | Merge $H$ and $d_v$ back together | `(B, T, d_model)` |
| **Final Linear Layer Output** | Project back to Hidden Size | `(B, T, d_model)` |

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=64, num_heads=4, d_ff=128):
        super().__init__()
        # 1. Multi-Head Attention components
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        
        # 2. Layer Normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # 3. Position-Wise Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        # x shape: [Batch_Size, Seq_Len, d_model]
        batch_size, seq_len, d_model = x.shape
        
        # --- STAGE 1: MULTI-HEAD SELF-ATTENTION ---SingleEncoderLayer
        # Project inputs to Queries, Keys, and Values
        # Split into heads and transpose to: [Batch_Size, Num_Heads, Seq_Len, d_k]
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Calculate Scaled Dot-Product Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)
        
        # Mix weights with Values and merge attention heads back together
        context = torch.matmul(attention_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        attn_output = self.out_linear(context)
        
        # First Residual Connection + LayerNorm
        x = self.norm1(x + attn_output)
        
        # --- STAGE 2: FEED-FORWARD NETWORK ---
        ffn_output = self.ffn(x)
        
        # Second Residual Connection + LayerNorm
        output = self.norm2(x + ffn_output)
        
        return output

We can define an encoder and pass data to it.

Notice that the output has the same dimensions as the input. This allows us to stack multiple encoderlayers together.

In [ ]:
batch_size = 2   # Number of sample sequences
seq_len = 5      # Length of the timeline/sentence
d_model = 64     # High-dimensional vector space size

# 1. Create a mock input tensor (e.g., already embedded data)
mock_input = torch.randn(batch_size, seq_len, d_model)

# 2. Initialize our custom Encoder Layer
# an Embedding module containing 10 tensors of size 100
encoder_block = EncoderLayer(d_model=d_model, num_heads=4, d_ff=128)

# 3. Pass the data through the Encoder
processed_output = encoder_block(mock_input)

print("=== SHAPE VERIFICATION ===")
print(f"Input Shape:  {mock_input.shape}")       # Expected: [2, 5, 64]
print(f"Output Shape: {processed_output.shape}")  # Expected: [2, 5, 64] (Shape is preserved!)